# Assignment 2 Geostrophic transport and relating AMOC series

Author: Hengxi Yang

Date: Sep 2026

## Part 1 Geostrophic transport

In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import sys
from pathlib import Path
import os
sys.path.append(str(Path.cwd().parent))

from correlation_trends.data_io import load_ts_gridded, load_amoc
from correlation_trends.geostrophy import interior_geostrophic_transport
from correlation_trends.simple_calc import print_timeseries_info
from correlation_trends.plotting import plot_umo_comparison
from correlation_trends.simple_calc import report_series_correlation_and_regression
from correlation_trends.plotting import plot_series_scatter_and_regression


### 1.1 Load hydrography

In [ ]:
# Because of the unstable server connnection, loading always fails, so I downloaded the dataset directly and put it in the `data` folder.
data_dir = str(Path.cwd().parent / "data")

if os.path.exists(os.path.join(data_dir, "ts_gridded.nc")):
    hydro_data = load_ts_gridded(data_dir=data_dir)
else:
    try:
        hydro_data = load_ts_gridded(data_dir=data_dir)
    except Exception as e:
        print(f"Server connection failed.")


# load official products
official_time_axis, sample_spacing_days, official_transport_dict = load_amoc()


### 1.2 Compute the upper mid-ocean geostrophic transport

In [ ]:
# compute UMO transport series
T_UMO = interior_geostrophic_transport(hydro_data)

Print time span and smapling pace (unit: day)

In [ ]:
print_timeseries_info(T_UMO, official_time_axis, sample_spacing_days)

### 1.3 Comparison of Calculated UMO Series and Official UMO Series

Overlay Map

In [ ]:
official_umo = official_transport_dict["TRANS_UMO"]

fig_comp, ax = plot_umo_comparison(
    time = T_UMO.TIME.values,
    calculated_umo = T_UMO.values,
    official_umo = official_umo,
)

fig_comp.savefig("../figures/Part1_Comparison_of_calculated_and_official_UMO_transport_series.png", dpi = 600, bbox_inches = "tight")


### 1.4 Correlation between Calculated UMO series and Official UMO series

In [ ]:
results = report_series_correlation_and_regression(
    x = T_UMO.values,
    y = official_transport_dict["TRANS_UMO"],
    name_x = "Calculated UMO",
    name_y = "Official UMO",
)

### 1.5 Scatter Plot and Linear Regression

In [ ]:
fig_reg, ax = plot_series_scatter_and_regression(
    results = results,
    name_x = "Calculated UMO Transport (Sv)",
    name_y = "Official UMO Transport (Sv)",
    title = "UMO Transport Regression Analysis",
    dpi = 600,
)

fig_reg.savefig("../figures/Part1_UMO_regression_analysis.png", dpi = 600, bbox_inches = "tight")

## Part 2A Single series: seasonal cycle and trend (MOC & Trans_UMO)

In [ ]:
import numpy as np

from correlation_trends.data_io import load_amoc
from correlation_trends.simple_calc import prepare_seasonal_and_deseasonalised_series
from correlation_trends.plotting import plot_monthly_climatology, plot_raw_vs_deseasonalised
from correlation_trends.correlation import autocorr, integral_timescale
from correlation_trends.plotting import plot_autocorrelation
from correlation_trends.correlation import effective_dof
from correlation_trends.trends import trend_with_significance
from correlation_trends.simple_calc import report_trend_results

In [ ]:
# Load MOC and Trans_UMO
time, dt, series = load_amoc()
trans_moc = series["MOC"]
trans_umo = series["TRANS_UMO"]

### 2A.1 Monthly Climatology and Deseasonalised Series

In [ ]:
# deseasonalise
deseasoned_transports = prepare_seasonal_and_deseasonalised_series(
    time = time,
    trans_moc = series["MOC"],
    trans_umo = series["TRANS_UMO"],
)

In [ ]:
# seasnal cycle plot
fig_clim, _ = plot_monthly_climatology(
    clim_moc = deseasoned_transports["clim_moc"],
    clim_umo = deseasoned_transports["clim_umo"],
)

fig_clim.savefig("../figures/Part2A.1_MOC_and_UMO_Monthly_Climatology.png", dpi = 600, bbox_inches = "tight")

In [ ]:
# row and deseasoned series
fig_raw_vs_deseas, _ = plot_raw_vs_deseasonalised(
    time = time,
    da_moc = deseasoned_transports["da_moc"],
    deseason_moc = deseasoned_transports["deseason_moc"],
    da_umo = deseasoned_transports["da_umo"],
    deseason_umo = deseasoned_transports["deseason_umo"],
)

fig_raw_vs_deseas.savefig("../figures/Part2A.1_MOC_and_UMO_raw_and_deseasonalised_series.png", dpi = 600, bbox_inches = "tight")

### 2A.2 Autocorrelation with T*

In [ ]:
# compute autocorrelation array and T*

r_moc = autocorr(deseasoned_transports["deseason_moc"].values)
t_int_moc = integral_timescale(deseasoned_transports["deseason_moc"].values, dt = dt) # dt is from load_amoc()

r_umo = autocorr(deseasoned_transports["deseason_umo"].values)
t_int_umo = integral_timescale(deseasoned_transports["deseason_umo"].values, dt = dt)

# calculate lag in days
lags_moc = np.arange(len(r_moc)) * dt
lags_umo = np.arange(len(r_umo)) * dt

print(f"T* of MOC is {t_int_moc} days") #days
print(f"T* of UMO is {t_int_umo} days")


In [ ]:
# plot autocorrelation (MOC and UMO)

# MOC
fig_moc, ax_moc = plot_autocorrelation(
    lags = lags_moc,
    r = r_moc,
    t_int = t_int_moc,
    var_name = "MOC",
    color = "salmon",
    max_lag = 365.0*2,
)

fig_moc.savefig("../figures/Part2A.2_MOC_autocorrelation_with_t_int.png", dpi = 600, bbox_inches = "tight")

# UMO
fig_umo,ax_umo = plot_autocorrelation(
    lags = lags_umo,
    r = r_umo,
    t_int = t_int_umo,
    var_name = "UMO",
    color = "tab:blue",
    max_lag = 365.0*2,
)

fig_umo.savefig("../figures/Part2A.2_UMO_autocorrelation_with_t_int.png", dpi = 600, bbox_inches = "tight")


### 2A.3 Linear Trend Info Based on $N_{eff}$ 

In [ ]:
# compute the real N_eff (not the linear fit N_eff)
edof_moc = effective_dof(deseasoned_transports["deseason_moc"].values, dt = dt)
edof_umo = effective_dof(deseasoned_transports["deseason_umo"].values, dt = dt)

print(f"MOC EDOF is {edof_moc}")
print(f"UMO EDOF is {edof_umo}")

In [ ]:
# set time axis in years
time_days = (deseasoned_transports["deseason_moc"].TIME.values - deseasoned_transports["deseason_moc"].TIME.values[0]) / np.timedelta64(1, "D")
time_years = time_days / 365.25

In [ ]:
# linear trend
trend_moc = trend_with_significance(
    time_years, deseasoned_transports["deseason_moc"].values, dt=dt
)
trend_umo = trend_with_significance(
    time_years, deseasoned_transports["deseason_umo"].values, dt=dt
)

In [ ]:
# report the info
report_trend_results(trend_moc, name = "MOC")
report_trend_results(trend_umo, name = "UMO")

## Part 2B Cross-correlation

- **I chose 26°N vs 47°N MOC_SIGMA0**

In [ ]:
import matplotlib.pyplot as plt
from correlation_trends.data_io import load_47n, load_moc_sigma0_26n
from correlation_trends.simple_calc import match_series
from correlation_trends.plotting import plot_matched_series_comparison
from correlation_trends.seasonal import remove_seasonal_cycle
from correlation_trends.correlation import cross_correlation
from correlation_trends.plotting import plot_cross_relation_comparison, plot_lagged_scatter
from correlation_trends.correlation import cross_correlation_significance

In [ ]:
# load series
time_26n, moc_26n = load_moc_sigma0_26n(data_dir=data_dir)
time_47n, moc_47n = load_47n()

# info
print(f"26°N time span is from {time_26n[0]} to {time_26n[-1]} (Points: {len(time_26n)})")
print(f"47°N time span is from {time_47n[0]} to {time_47n[-1]} (Points: {len(time_47n)})")

### 2B.1 Match the 2 series

In [ ]:
# match 2 series
da_26n_matched, da_47n_matched = match_series(
    time1 = time_26n, x1 = moc_26n,
    time2 = time_47n, x2 = moc_47n,
    freq="ME" # month end
)

In [ ]:
# before/after figure
fig_match, axes = plot_matched_series_comparison(
    time_raw1 = time_26n, x_raw1 = moc_26n,
    time_raw2 = time_47n, x_raw2 = moc_47n,
    time_m1 = da_26n_matched.TIME.values, x_m1 = da_26n_matched.values,
    time_m2 = da_47n_matched.TIME.values, x_m2 = da_47n_matched.values,
    name1 = "26°N MOC", name2 = "47°N MOC"
)

fig_match.savefig("../figures/Part2B.1_26MOC_and_47_MOC_before_and_after_matching.png", dpi = 600, bbox_inches = "tight")

### 2B.2 Cross-correlation

In [ ]:
# deseasonalise
da_26n_deseason = remove_seasonal_cycle(da_26n_matched)
da_47n_deseason = remove_seasonal_cycle(da_47n_matched)

In [ ]:
# cross-correlate
lags, r_raw = cross_correlation(da_26n_matched.values, da_47n_matched.values)
_, r_deseason = cross_correlation(da_26n_deseason.values, da_47n_deseason.values)

In [ ]:
# find peak lag and r-val
peak_idx_raw = np.argmax(r_raw) #the index of max value in r_raw
peak_lag_raw = lags[peak_idx_raw] # check the lag matched to max value
peak_r_raw = r_raw[peak_idx_raw] # get the r value matched to the peak lag

peak_idx_deseason = np.argmax(r_deseason)
peak_lag_deseason = lags[peak_idx_deseason]
peak_r_deseason = r_deseason[peak_idx_deseason]

print(f"Raw peak lag is {peak_lag_raw} months, r = {peak_r_raw:.3f}")
print(f"Deseasonalised peak lag is {peak_lag_deseason} months, r = {peak_r_deseason:.3f}")

In [ ]:
fig_cc, ax_cc = plot_cross_relation_comparison(
    lags, r_raw, r_deseason,
    peak_raw = (peak_lag_raw, peak_r_raw),
    peak_deseason = (peak_lag_deseason, peak_r_deseason),
    var_names = ("26°N MOC", "47°N MOC")
)

fig_cc.savefig("../figures/Part2B.2_26MOC_and_47_MOC_cross_correlation.png", dpi = 600, bbox_inches = "tight")

In [ ]:
# plot scatter at the peak lag
fig_scatter, (ax1, ax2) = plt.subplots(1, 2,figsize = (12, 6))

# raw
plot_lagged_scatter(
    da_26n_matched.values, da_47n_matched.values,
    peak_lag = int(peak_lag_raw),
    var_names = ("26°N MOC", "47°N MOC"),
    ax = ax1,
    title_suffix = "Raw"
)

# deseasonalised
plot_lagged_scatter(
    da_26n_deseason.values, da_47n_deseason.values,
    peak_lag = int(peak_lag_deseason),
    var_names  =("26°N MOC", "47°N MOC"),
    ax = ax2,
    title_suffix  ="Deseasonalised"
)

fig_scatter.savefig("../figures/Part2B.2_26MOC_and_47_MOC_raw_and_deseasonalised_scatter_at_the_peak_lag.png", dpi = 600, bbox_inches = "tight")

In [ ]:
### 2B.3 Effective sample size & significance
sig_result = cross_correlation_significance(
    da_26n_deseason.values, 
    da_47n_deseason.values, 
    r_val=peak_r_deseason, 
    dt=1.0, 
    alpha=0.05
)

print(f"N_eff = {sig_result['n_eff']:.3f}")
print(f"95% Critical r is {sig_result['r_crit']:.3f}")
print(f"Peak ris {peak_r_deseason:.3f}")
print(f"Statistically Significant: {sig_result['significant']}")

# Part 3 Depth Sensitivity Analysis

In [ ]:
from correlation_trends.simple_calc import depth_sensitivity
from correlation_trends.plotting import plot_depth_sensitivity

In [ ]:
# 3 series from diff depths integration, UMO

umo_dict = depth_sensitivity(
    ds=hydro_data, 
    official_time=official_time_axis, 
    official_umo=official_transport_dict['TRANS_UMO'], 
    depths=[700.0, 1000.0, 1100.0]
)

In [ ]:
# plot sensitivity figure with official UMO transport series
fig_ds, ax_ds = plot_depth_sensitivity(
    umo_dict=umo_dict,
    official_time=official_time_axis,
    official_umo=official_transport_dict['TRANS_UMO']
)

fig_ds.savefig("../figures/Part3_depth_sensitivity_of_26UMO_transport.png", dpi = 600, bbox_inches = "tight")